Rusty Bargain used car sales service is developing an app to attract new customers. In that app, you can quickly find out the market value of your car. You have access to historical data: technical specifications, trim versions, and prices. You need to build the model to determine the value. 

Rusty Bargain is interested in:

- the quality of the prediction;
- the speed of the prediction;
- the time required for training

In [1]:
%matplotlib inline
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# 1. Data preparation

### Downloading

In [2]:
data = pd.read_csv('autos.csv')
print(data.shape)
data.head()

(354369, 16)


,DateCrawled,Price,VehicleType,RegistrationYear,Gearbox,Power,Model,Kilometer,RegistrationMonth,FuelType,Brand,NotRepaired,DateCreated,NumberOfPictures,PostalCode,LastSeen
0,2016-03-24 11:52:17,480,NaN,1993,manual,0,golf,150000,0,petrol,volkswagen,NaN,2016-03-24 00:00:00,0,70435,2016-04-07 03:16:57
1,2016-03-24 10:58:45,18300,coupe,2011,manual,190,NaN,125000,5,gasoline,audi,yes,2016-03-24 00:00:00,0,66954,2016-04-07 01:46:50
2,2016-03-14 12:52:21,9800,suv,2004,auto,163,grand,125000,8,gasoline,jeep,NaN,2016-03-14 00:00:00,0,90480,2016-04-05 12:47:46
3,2016-03-17 16:54:04,1500,small,2001,manual,75,golf,150000,6,petrol,volkswagen,no,2016-03-17 00:00:00,0,91074,2016-03-17 17:40:17
4,2016-03-31 17:25:20,3600,small,2008,manual,69,fabia,90000,7,gasoline,skoda,no,2016-03-31 00:00:00,0,60437,2016-04-06 10:17:21


### Preprocessing

Unnecessary features (can't be used in the product):

- dates
- zip-code

In [3]:
data = data.drop(['DateCrawled', 'DateCreated', 'PostalCode', 'LastSeen'], axis=1)
data.head()

,Price,VehicleType,RegistrationYear,Gearbox,Power,Model,Kilometer,RegistrationMonth,FuelType,Brand,NotRepaired,NumberOfPictures
0,480,NaN,1993,manual,0,golf,150000,0,petrol,volkswagen,NaN,0
1,18300,coupe,2011,manual,190,NaN,125000,5,gasoline,audi,yes,0
2,9800,suv,2004,auto,163,grand,125000,8,gasoline,jeep,NaN,0
3,1500,small,2001,manual,75,golf,150000,6,petrol,volkswagen,no,0
4,3600,small,2008,manual,69,fabia,90000,7,gasoline,skoda,no,0


In [4]:
data['NumberOfPictures'].value_counts()

0    354369
Name: NumberOfPictures, dtype: int64

Deleting constant feature

In [5]:
data = data.drop(['NumberOfPictures'], axis=1)
data.head()

,Price,VehicleType,RegistrationYear,Gearbox,Power,Model,Kilometer,RegistrationMonth,FuelType,Brand,NotRepaired
0,480,NaN,1993,manual,0,golf,150000,0,petrol,volkswagen,NaN
1,18300,coupe,2011,manual,190,NaN,125000,5,gasoline,audi,yes
2,9800,suv,2004,auto,163,grand,125000,8,gasoline,jeep,NaN
3,1500,small,2001,manual,75,golf,150000,6,petrol,volkswagen,no
4,3600,small,2008,manual,69,fabia,90000,7,gasoline,skoda,no


In [6]:
data.describe()

,Price,RegistrationYear,Power,Kilometer,RegistrationMonth
count,354369.000000,354369.000000,354369.000000,354369.000000,354369.000000
mean,4416.656776,2004.234448,110.094337,128211.172535,5.714645
std,4514.158514,90.227958,189.850405,37905.341530,3.726421
min,0.000000,1000.000000,0.000000,5000.000000,0.000000
25%,1050.000000,1999.000000,69.000000,125000.000000,3.000000
50%,2700.000000,2003.000000,105.000000,150000.000000,6.000000
75%,6400.000000,2008.000000,143.000000,150000.000000,9.000000
max,20000.000000,9999.000000,20000.000000,150000.000000,12.000000


The registration year of 1000 and 9999 is clearly incorrect, delete entries with incorrect values.

Power cannot be equal to 0.

In [7]:
data = data[data['RegistrationYear'] < 2050]
data = data[data['RegistrationYear'] > 1900]
data = data[data['Power'] != 0]

data.reset_index()
data.shape

(314100, 11)

In [8]:
data.isna().sum(axis=0)

Price                    0
VehicleType          22818
RegistrationYear         0
Gearbox               6501
Power                    0
Model                13395
Kilometer                0
RegistrationMonth        0
FuelType             21212
Brand                    0
NotRepaired          49708
dtype: int64

Missing values are only in categorical columns. They can be filled with a new value "unknown".

In [9]:
data = data.fillna('unknown')
data.isna().sum(axis=0)

Price                0
VehicleType          0
RegistrationYear     0
Gearbox              0
Power                0
Model                0
Kilometer            0
RegistrationMonth    0
FuelType             0
Brand                0
NotRepaired          0
dtype: int64

### Feature encoding

In [10]:
categorical_features = [
    'VehicleType',
    'Gearbox', 
    'Model',
    'FuelType', 
    'Brand',
    'NotRepaired', 
]

**OHE-encoding**

Here, some features will have to be removed due to the large number of values

In [11]:
for feature in categorical_features:
    print(data[feature].value_counts())

sedan          84852
small          71522
wagon          60508
bus            26530
unknown        22818
convertible    19034
coupe          15078
suv            11115
other           2643
Name: VehicleType, dtype: int64
manual     246097
auto        61502
unknown      6501
Name: Gearbox, dtype: int64
golf                  26760
other                 21248
3er                   18226
unknown               13395
polo                  11455
corsa                 10798
astra                  9698
passat                 9263
a4                     9171
c_klasse               7568
5er                    7369
e_klasse               6147
a3                     5630
focus                  5337
a6                     5178
fiesta                 5048
2_reihe                4494
transporter            4326
twingo                 3998
fortwo                 3914
vectra                 3734
a_klasse               3654
1er                    3391
mondeo                 3168
3_reihe                31

In [12]:
data_ohe = data.drop(['Model', 'Brand'], axis=1)
data_ohe = pd.get_dummies(data_ohe)
print(data_ohe.shape)
data_ohe.head()

(314100, 28)


,Price,RegistrationYear,Power,Kilometer,RegistrationMonth,VehicleType_bus,VehicleType_convertible,VehicleType_coupe,VehicleType_other,VehicleType_sedan,...,FuelType_electric,FuelType_gasoline,FuelType_hybrid,FuelType_lpg,FuelType_other,FuelType_petrol,FuelType_unknown,NotRepaired_no,NotRepaired_unknown,NotRepaired_yes
1,18300,2011,190,125000,5,0,0,1,0,0,...,0,1,0,0,0,0,0,0,0,1
2,9800,2004,163,125000,8,0,0,0,0,0,...,0,1,0,0,0,0,0,0,1,0
3,1500,2001,75,150000,6,0,0,0,0,0,...,0,0,0,0,0,1,0,1,0,0
4,3600,2008,69,90000,7,0,0,0,0,0,...,0,1,0,0,0,0,0,1,0,0
5,650,1995,102,150000,10,0,0,0,0,1,...,0,0,0,0,0,1,0,0,0,1


Ordinal encoding

In [13]:
from sklearn.preprocessing import OrdinalEncoder

data[categorical_features] = OrdinalEncoder().fit_transform(data[categorical_features])

data.head()

,Price,VehicleType,RegistrationYear,Gearbox,Power,Model,Kilometer,RegistrationMonth,FuelType,Brand,NotRepaired
1,18300,2.0,2011,1.0,190,227.0,125000,5,2.0,1.0,2.0
2,9800,6.0,2004,0.0,163,117.0,125000,8,2.0,14.0,1.0
3,1500,5.0,2001,1.0,75,116.0,150000,6,6.0,38.0,0.0
4,3600,5.0,2008,1.0,69,101.0,90000,7,2.0,31.0,0.0
5,650,4.0,1995,1.0,102,11.0,150000,10,6.0,2.0,2.0


### Split into train-validation-test

In [14]:
from sklearn.model_selection import train_test_split

index_train_valid, index_test = train_test_split(data.index, test_size=0.2, random_state=12345)
index_train, index_valid = train_test_split(index_train_valid, test_size=0.25, random_state=54321)

data_train = data.loc[index_train]
data_valid = data.loc[index_valid]
data_test = data.loc[index_test]

data_ohe_train = data_ohe.loc[index_train]
data_ohe_valid = data_ohe.loc[index_valid]
data_ohe_test = data_ohe.loc[index_test]

print(data_train.shape)
print(data_valid.shape)
print(data_test.shape)

print(data_ohe_train.shape)
print(data_ohe_valid.shape)
print(data_ohe_test.shape)

(188460, 11)
(62820, 11)
(62820, 11)
(188460, 28)
(62820, 28)
(62820, 28)


# 2. Model training

In [15]:
from sklearn.metrics import mean_squared_error

def rmse(y, a):
    return mean_squared_error(y, a)**0.5

Constant model

In [16]:
pred_mean = np.ones(data['Price'].shape) * data['Price'].mean()
print(rmse(data['Price'], pred_mean))

4590.240919951061


### Models with OHE

In [17]:
features_train = data_ohe_train.drop(['Price'], axis=1)
target_train = data_ohe_train['Price']
features_valid = data_ohe_valid.drop(['Price'], axis=1)
target_valid = data_ohe_valid['Price']
features_test = data_ohe_test.drop(['Price'], axis=1)
target_test = data_ohe_test['Price']

**Linear regression**

In [18]:
%%time

from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(features_train, target_train)

CPU times: user 358 ms, sys: 66.9 ms, total: 425 ms
Wall time: 398 ms


In [19]:
%%time

pred_train = model.predict(features_train)
pred_valid = model.predict(features_valid)
pred_test = model.predict(features_test)

CPU times: user 57.9 ms, sys: 32 ms, total: 89.9 ms
Wall time: 111 ms


In [20]:
print("Train RMSE:", rmse(target_train, pred_train).round(5))
print("Valid RMSE:", rmse(target_valid, pred_valid).round(5))
print("Test RMSE: ", rmse(target_test, pred_test).round(5))

Train RMSE: 3238.30929
Valid RMSE: 3216.58303
Test RMSE:  3223.39036


### Models with ordinal encoding

In [21]:
features_train = data_train.drop(['Price'], axis=1)
target_train = data_train['Price']
features_valid = data_valid.drop(['Price'], axis=1)
target_valid = data_valid['Price']
features_test = data_test.drop(['Price'], axis=1)
target_test = data_test['Price']

**Random forrest**

In [22]:
from sklearn.ensemble import RandomForestRegressor

for depth in [1, 2, 4, 6, 8, None]:
    model = RandomForestRegressor(max_depth=depth, n_estimators=100)
    model.fit(features_train, target_train)
    
    pred_train = model.predict(features_train)
    pred_valid = model.predict(features_valid)
    print("Depth:", depth)
    print("Train RMSE:", rmse(target_train, pred_train).round(5))
    print("Valid RMSE:", rmse(target_valid, pred_valid).round(5))

Depth: 1
Train RMSE: 3790.22028
Valid RMSE: 3782.28954
Depth: 2
Train RMSE: 3329.79646
Valid RMSE: 3318.93867
Depth: 4
Train RMSE: 2691.49521
Valid RMSE: 2677.68587
Depth: 6
Train RMSE: 2332.27542
Valid RMSE: 2333.8505
Depth: 8
Train RMSE: 2099.20864
Valid RMSE: 2124.48877
Depth: None
Train RMSE: 771.00994
Valid RMSE: 1729.71112


In [23]:
%%time

from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(n_estimators=100, max_depth=None)
model.fit(features_train, target_train)

CPU times: user 1min 16s, sys: 1.86 s, total: 1min 18s
Wall time: 1min 27s


In [24]:
%%time

pred_train = model.predict(features_train)
pred_valid = model.predict(features_valid)
pred_test = model.predict(features_test)

CPU times: user 17 s, sys: 257 ms, total: 17.3 s
Wall time: 19.4 s


In [25]:
print("Train RMSE:", rmse(target_train, pred_train).round(5))
print("Valid RMSE:", rmse(target_valid, pred_valid).round(5))
print("Test RMSE: ", rmse(target_test, pred_test).round(5))

Train RMSE: 771.64466
Valid RMSE: 1732.70045
Test RMSE:  1728.78498


**Gradient boosting LightGBM**

In [26]:
%%time

import lightgbm as lgb

model = lgb.LGBMRegressor(num_iterations=1000, vebose=1, metric='rmse')
model.fit(features_train, target_train, 
          eval_set=(features_valid, target_valid),
          categorical_feature=categorical_features)

/usr/local/lib/python3.7/site-packages/lightgbm/__init__.py:46: UserWarning: Starting from version 2.2.1, the library file in distribution wheels for macOS is built by the Apple Clang (Xcode_8.3.3) compiler.
This means that in case of installing LightGBM from PyPI via the ``pip install lightgbm`` command, you don't need to install the gcc compiler anymore.
Instead of that, you need to install the OpenMP library, which is required for running LightGBM on the system with the Apple Clang compiler.
You can install the OpenMP library by the following command: ``brew install libomp``.
  "You can install the OpenMP library by the following command: ``brew install libomp``.", UserWarning)
/usr/local/lib/python3.7/site-packages/lightgbm/engine.py:118: UserWarning: Found `num_iterations` in params. Will use it instead of argument
  warnings.warn("Found `{}` in params. Will use it instead of argument".format(alias))
/usr/local/lib/python3.7/site-packages/lightgbm/basic.py:1209: UserWarning: categ

[1]	valid_0's rmse: 4264.85
[2]	valid_0's rmse: 3979.75
[3]	valid_0's rmse: 3730.63
[4]	valid_0's rmse: 3509.73
[5]	valid_0's rmse: 3314.84
[6]	valid_0's rmse: 3143.37
[7]	valid_0's rmse: 2990.09
[8]	valid_0's rmse: 2852.56
[9]	valid_0's rmse: 2734.12
[10]	valid_0's rmse: 2626.74
[11]	valid_0's rmse: 2535
[12]	valid_0's rmse: 2451.28
[13]	valid_0's rmse: 2380.15
[14]	valid_0's rmse: 2316.23
[15]	valid_0's rmse: 2258.4
[16]	valid_0's rmse: 2209.11
[17]	valid_0's rmse: 2163.7
[18]	valid_0's rmse: 2125.11
[19]	valid_0's rmse: 2090.67
[20]	valid_0's rmse: 2060.78
[21]	valid_0's rmse: 2033.48
[22]	valid_0's rmse: 2010.38
[23]	valid_0's rmse: 1989.39
[24]	valid_0's rmse: 1970.55
[25]	valid_0's rmse: 1954.17
[26]	valid_0's rmse: 1938.74
[27]	valid_0's rmse: 1925.01
[28]	valid_0's rmse: 1912.61
[29]	valid_0's rmse: 1901.93
[30]	valid_0's rmse: 1890.83
[31]	valid_0's rmse: 1882.13
[32]	valid_0's rmse: 1874.62
[33]	valid_0's rmse: 1866.91
[34]	valid_0's rmse: 1859.39
[35]	valid_0's rmse: 1853.48

[291]	valid_0's rmse: 1690.74
[292]	valid_0's rmse: 1690.63
[293]	valid_0's rmse: 1690.69
[294]	valid_0's rmse: 1690.63
[295]	valid_0's rmse: 1690.61
[296]	valid_0's rmse: 1690.57
[297]	valid_0's rmse: 1690.4
[298]	valid_0's rmse: 1689.92
[299]	valid_0's rmse: 1689.88
[300]	valid_0's rmse: 1689.85
[301]	valid_0's rmse: 1689.72
[302]	valid_0's rmse: 1689.68
[303]	valid_0's rmse: 1689.19
[304]	valid_0's rmse: 1689.04
[305]	valid_0's rmse: 1688.95
[306]	valid_0's rmse: 1688.82
[307]	valid_0's rmse: 1688.61
[308]	valid_0's rmse: 1688.51
[309]	valid_0's rmse: 1688.43
[310]	valid_0's rmse: 1688.33
[311]	valid_0's rmse: 1688.35
[312]	valid_0's rmse: 1688.23
[313]	valid_0's rmse: 1688.23
[314]	valid_0's rmse: 1688.2
[315]	valid_0's rmse: 1688.09
[316]	valid_0's rmse: 1687.97
[317]	valid_0's rmse: 1688.05
[318]	valid_0's rmse: 1688.09
[319]	valid_0's rmse: 1687.99
[320]	valid_0's rmse: 1687.67
[321]	valid_0's rmse: 1687.31
[322]	valid_0's rmse: 1687.15
[323]	valid_0's rmse: 1686.81
[324]	valid_

[571]	valid_0's rmse: 1665.98
[572]	valid_0's rmse: 1665.95
[573]	valid_0's rmse: 1665.99
[574]	valid_0's rmse: 1666.05
[575]	valid_0's rmse: 1666.02
[576]	valid_0's rmse: 1665.96
[577]	valid_0's rmse: 1665.76
[578]	valid_0's rmse: 1665.57
[579]	valid_0's rmse: 1665.51
[580]	valid_0's rmse: 1665.45
[581]	valid_0's rmse: 1665.39
[582]	valid_0's rmse: 1665.44
[583]	valid_0's rmse: 1665.49
[584]	valid_0's rmse: 1665.42
[585]	valid_0's rmse: 1665.49
[586]	valid_0's rmse: 1665.49
[587]	valid_0's rmse: 1665.43
[588]	valid_0's rmse: 1665.47
[589]	valid_0's rmse: 1665.45
[590]	valid_0's rmse: 1665.41
[591]	valid_0's rmse: 1665.36
[592]	valid_0's rmse: 1665.41
[593]	valid_0's rmse: 1665.37
[594]	valid_0's rmse: 1665.36
[595]	valid_0's rmse: 1665.28
[596]	valid_0's rmse: 1665.26
[597]	valid_0's rmse: 1665.28
[598]	valid_0's rmse: 1665.28
[599]	valid_0's rmse: 1665.31
[600]	valid_0's rmse: 1665.31
[601]	valid_0's rmse: 1665.34
[602]	valid_0's rmse: 1665.31
[603]	valid_0's rmse: 1665.31
[604]	vali

[870]	valid_0's rmse: 1653.84
[871]	valid_0's rmse: 1653.67
[872]	valid_0's rmse: 1653.54
[873]	valid_0's rmse: 1653.48
[874]	valid_0's rmse: 1653.48
[875]	valid_0's rmse: 1653.48
[876]	valid_0's rmse: 1653.51
[877]	valid_0's rmse: 1653.52
[878]	valid_0's rmse: 1653.48
[879]	valid_0's rmse: 1653.41
[880]	valid_0's rmse: 1653.4
[881]	valid_0's rmse: 1653.44
[882]	valid_0's rmse: 1653.44
[883]	valid_0's rmse: 1653.42
[884]	valid_0's rmse: 1653.41
[885]	valid_0's rmse: 1653.42
[886]	valid_0's rmse: 1653.4
[887]	valid_0's rmse: 1653.41
[888]	valid_0's rmse: 1653.39
[889]	valid_0's rmse: 1653.32
[890]	valid_0's rmse: 1653.23
[891]	valid_0's rmse: 1653.18
[892]	valid_0's rmse: 1653.17
[893]	valid_0's rmse: 1653.1
[894]	valid_0's rmse: 1653.1
[895]	valid_0's rmse: 1653.12
[896]	valid_0's rmse: 1653.11
[897]	valid_0's rmse: 1653.07
[898]	valid_0's rmse: 1653.06
[899]	valid_0's rmse: 1653.03
[900]	valid_0's rmse: 1652.95
[901]	valid_0's rmse: 1652.89
[902]	valid_0's rmse: 1652.87
[903]	valid_0'

In [27]:
%%time

pred_train = model.predict(features_train)
pred_valid = model.predict(features_valid)
pred_test = model.predict(features_test)

CPU times: user 32.4 s, sys: 267 ms, total: 32.7 s
Wall time: 12.2 s


In [28]:
print("Train RMSE:", rmse(target_train, pred_train).round(5))
print("Valid RMSE:", rmse(target_valid, pred_valid).round(5))
print("Test RMSE: ", rmse(target_test, pred_test).round(5))

Train RMSE: 1410.9156
Valid RMSE: 1650.6038
Test RMSE:  1641.91689


In [29]:
%%time

import lightgbm as lgb

model = lgb.LGBMRegressor(num_iterations=1000, vebose=1, metric='rmse')
model.fit(features_train, target_train, 
          eval_set=(features_valid, target_valid))

/usr/local/lib/python3.7/site-packages/lightgbm/engine.py:118: UserWarning: Found `num_iterations` in params. Will use it instead of argument
  warnings.warn("Found `{}` in params. Will use it instead of argument".format(alias))


[1]	valid_0's rmse: 4268.81
[2]	valid_0's rmse: 3988.72
[3]	valid_0's rmse: 3745.28
[4]	valid_0's rmse: 3533.96
[5]	valid_0's rmse: 3344.79
[6]	valid_0's rmse: 3180.15
[7]	valid_0's rmse: 3038.53
[8]	valid_0's rmse: 2913.76
[9]	valid_0's rmse: 2800.89
[10]	valid_0's rmse: 2703.65
[11]	valid_0's rmse: 2623.15
[12]	valid_0's rmse: 2548.09
[13]	valid_0's rmse: 2484.49
[14]	valid_0's rmse: 2428.51
[15]	valid_0's rmse: 2376.84
[16]	valid_0's rmse: 2330.6
[17]	valid_0's rmse: 2291.97
[18]	valid_0's rmse: 2256.88
[19]	valid_0's rmse: 2225.96
[20]	valid_0's rmse: 2197.94
[21]	valid_0's rmse: 2172.21
[22]	valid_0's rmse: 2146.89
[23]	valid_0's rmse: 2125.14
[24]	valid_0's rmse: 2105.92
[25]	valid_0's rmse: 2085.68
[26]	valid_0's rmse: 2069.76
[27]	valid_0's rmse: 2055.27
[28]	valid_0's rmse: 2041.37
[29]	valid_0's rmse: 2030.06
[30]	valid_0's rmse: 2019.88
[31]	valid_0's rmse: 2009.93
[32]	valid_0's rmse: 1999.63
[33]	valid_0's rmse: 1990.62
[34]	valid_0's rmse: 1984.02
[35]	valid_0's rmse: 197

[295]	valid_0's rmse: 1732.08
[296]	valid_0's rmse: 1731.86
[297]	valid_0's rmse: 1731.49
[298]	valid_0's rmse: 1731.35
[299]	valid_0's rmse: 1731.23
[300]	valid_0's rmse: 1730.98
[301]	valid_0's rmse: 1730.86
[302]	valid_0's rmse: 1730.91
[303]	valid_0's rmse: 1730.76
[304]	valid_0's rmse: 1730.6
[305]	valid_0's rmse: 1730.4
[306]	valid_0's rmse: 1730.18
[307]	valid_0's rmse: 1729.97
[308]	valid_0's rmse: 1729.85
[309]	valid_0's rmse: 1729.48
[310]	valid_0's rmse: 1729.15
[311]	valid_0's rmse: 1728.73
[312]	valid_0's rmse: 1728.68
[313]	valid_0's rmse: 1728.45
[314]	valid_0's rmse: 1728.38
[315]	valid_0's rmse: 1728.26
[316]	valid_0's rmse: 1727.92
[317]	valid_0's rmse: 1727.77
[318]	valid_0's rmse: 1727.64
[319]	valid_0's rmse: 1727.68
[320]	valid_0's rmse: 1727.31
[321]	valid_0's rmse: 1727.03
[322]	valid_0's rmse: 1726.88
[323]	valid_0's rmse: 1726.7
[324]	valid_0's rmse: 1726.6
[325]	valid_0's rmse: 1726.54
[326]	valid_0's rmse: 1726.36
[327]	valid_0's rmse: 1726.25
[328]	valid_0'

[595]	valid_0's rmse: 1695.29
[596]	valid_0's rmse: 1695.11
[597]	valid_0's rmse: 1695.12
[598]	valid_0's rmse: 1695.03
[599]	valid_0's rmse: 1694.96
[600]	valid_0's rmse: 1694.92
[601]	valid_0's rmse: 1694.83
[602]	valid_0's rmse: 1694.87
[603]	valid_0's rmse: 1694.68
[604]	valid_0's rmse: 1694.6
[605]	valid_0's rmse: 1694.45
[606]	valid_0's rmse: 1694.43
[607]	valid_0's rmse: 1694.49
[608]	valid_0's rmse: 1694.53
[609]	valid_0's rmse: 1694.55
[610]	valid_0's rmse: 1694.44
[611]	valid_0's rmse: 1694.41
[612]	valid_0's rmse: 1694.35
[613]	valid_0's rmse: 1694.24
[614]	valid_0's rmse: 1694.17
[615]	valid_0's rmse: 1694.07
[616]	valid_0's rmse: 1694.12
[617]	valid_0's rmse: 1694.15
[618]	valid_0's rmse: 1693.9
[619]	valid_0's rmse: 1693.9
[620]	valid_0's rmse: 1693.89
[621]	valid_0's rmse: 1693.86
[622]	valid_0's rmse: 1693.67
[623]	valid_0's rmse: 1693.49
[624]	valid_0's rmse: 1693.46
[625]	valid_0's rmse: 1693.37
[626]	valid_0's rmse: 1693.32
[627]	valid_0's rmse: 1693.33
[628]	valid_0

[879]	valid_0's rmse: 1678.38
[880]	valid_0's rmse: 1678.37
[881]	valid_0's rmse: 1678.4
[882]	valid_0's rmse: 1678.33
[883]	valid_0's rmse: 1678.34
[884]	valid_0's rmse: 1678.24
[885]	valid_0's rmse: 1678.22
[886]	valid_0's rmse: 1678.21
[887]	valid_0's rmse: 1678.19
[888]	valid_0's rmse: 1678.19
[889]	valid_0's rmse: 1678.14
[890]	valid_0's rmse: 1678.12
[891]	valid_0's rmse: 1678.09
[892]	valid_0's rmse: 1678.1
[893]	valid_0's rmse: 1678.06
[894]	valid_0's rmse: 1678.07
[895]	valid_0's rmse: 1678
[896]	valid_0's rmse: 1677.98
[897]	valid_0's rmse: 1677.79
[898]	valid_0's rmse: 1677.76
[899]	valid_0's rmse: 1677.71
[900]	valid_0's rmse: 1677.73
[901]	valid_0's rmse: 1677.69
[902]	valid_0's rmse: 1677.65
[903]	valid_0's rmse: 1677.67
[904]	valid_0's rmse: 1677.55
[905]	valid_0's rmse: 1677.45
[906]	valid_0's rmse: 1677.41
[907]	valid_0's rmse: 1677.48
[908]	valid_0's rmse: 1677.46
[909]	valid_0's rmse: 1677.49
[910]	valid_0's rmse: 1677.46
[911]	valid_0's rmse: 1677.43
[912]	valid_0's

In [30]:
%%time

pred_train = model.predict(features_train)
pred_valid = model.predict(features_valid)
pred_test = model.predict(features_test)

CPU times: user 25.6 s, sys: 219 ms, total: 25.8 s
Wall time: 9.01 s


In [31]:
print("Train RMSE:", rmse(target_train, pred_train).round(5))
print("Valid RMSE:", rmse(target_valid, pred_valid).round(5))
print("Test RMSE: ", rmse(target_test, pred_test).round(5))

Train RMSE: 1460.69366
Valid RMSE: 1674.35245
Test RMSE:  1657.71951


**Gradient boosting CatBoost**

In [32]:
%%time

from catboost import CatBoostRegressor

model = CatBoostRegressor(iterations=1000,
                          learning_rate=0.1,
                          cat_features=categorical_features,
                          metric_period=50)
model.fit(features_train, target_train, 
          eval_set=(features_valid, target_valid))

0:	learn: 6040.0153813	test: 6028.7735266	best: 6028.7735266 (0)	total: 375ms	remaining: 6m 14s
50:	learn: 1977.2979179	test: 1975.1906172	best: 1975.1906172 (50)	total: 12s	remaining: 3m 43s
100:	learn: 1878.4069481	test: 1884.6503300	best: 1884.6503300 (100)	total: 23.2s	remaining: 3m 26s
150:	learn: 1831.3260017	test: 1844.0234889	best: 1844.0234889 (150)	total: 33.8s	remaining: 3m 10s
200:	learn: 1801.3435718	test: 1819.9635823	best: 1819.9635823 (200)	total: 42.5s	remaining: 2m 48s
250:	learn: 1779.9786481	test: 1802.8899208	best: 1802.8899208 (250)	total: 53.3s	remaining: 2m 38s
300:	learn: 1765.2174567	test: 1793.7512949	best: 1793.7512949 (300)	total: 1m 2s	remaining: 2m 25s
350:	learn: 1752.0341631	test: 1785.2491052	best: 1785.2491052 (350)	total: 1m 14s	remaining: 2m 17s
400:	learn: 1739.9659470	test: 1776.7791369	best: 1776.7791369 (400)	total: 1m 24s	remaining: 2m 6s
450:	learn: 1730.1814284	test: 1770.9604790	best: 1770.9604790 (450)	total: 1m 35s	remaining: 1m 55s
500:	l

In [33]:
%%time

pred_train = model.predict(features_train)
pred_valid = model.predict(features_valid)
pred_test = model.predict(features_test)

CPU times: user 7.37 s, sys: 98.8 ms, total: 7.47 s
Wall time: 4.45 s


In [34]:
print("Train RMSE:", rmse(target_train, pred_train).round(5))
print("Valid RMSE:", rmse(target_valid, pred_valid).round(5))
print("Test RMSE: ", rmse(target_test, pred_test).round(5))

Train RMSE: 1662.79723
Valid RMSE: 1736.34861
Test RMSE:  1723.89198


In [35]:
%%time

from catboost import CatBoostRegressor

model = CatBoostRegressor(iterations=1000,
                          learning_rate=0.1,
                          metric_period=50)
model.fit(features_train, target_train, 
          eval_set=(features_valid, target_valid))

0:	learn: 6049.6962500	test: 6038.3281568	best: 6038.3281568 (0)	total: 39.5ms	remaining: 39.5s
50:	learn: 2038.2343375	test: 2040.3568612	best: 2040.3568612 (50)	total: 1.42s	remaining: 26.4s
100:	learn: 1921.3669448	test: 1929.7341784	best: 1929.7341784 (100)	total: 3.19s	remaining: 28.4s
150:	learn: 1867.7983558	test: 1881.4822962	best: 1881.4822962 (150)	total: 4.78s	remaining: 26.9s
200:	learn: 1835.9848684	test: 1854.2801221	best: 1854.2801221 (200)	total: 6.48s	remaining: 25.8s
250:	learn: 1810.6084616	test: 1833.5939341	best: 1833.5939341 (250)	total: 7.79s	remaining: 23.2s
300:	learn: 1790.0584162	test: 1818.6976854	best: 1818.6976854 (300)	total: 9.3s	remaining: 21.6s
350:	learn: 1776.3028147	test: 1809.7277425	best: 1809.7277425 (350)	total: 10.6s	remaining: 19.5s
400:	learn: 1762.9300386	test: 1800.8860436	best: 1800.8860436 (400)	total: 11.8s	remaining: 17.6s
450:	learn: 1750.7138791	test: 1794.0672023	best: 1794.0672023 (450)	total: 13.1s	remaining: 16s
500:	learn: 1740.1

In [36]:
%%time

pred_train = model.predict(features_train)
pred_valid = model.predict(features_valid)
pred_test = model.predict(features_test)

CPU times: user 1.23 s, sys: 11.1 ms, total: 1.24 s
Wall time: 813 ms


In [37]:
print("Train RMSE:", rmse(target_train, pred_train).round(5))
print("Valid RMSE:", rmse(target_valid, pred_valid).round(5))
print("Test RMSE: ", rmse(target_test, pred_test).round(5))

Train RMSE: 1663.00123
Valid RMSE: 1746.32837
Test RMSE:  1734.82601


# 3. Model analysis

| Model | RMSE | Prediction time, с | Training time, with |
|----|----|-------|--------|
| Линейная регрессия | 3220 | 0.156 | 0.4 |
| LightGBM(1000) | 1640 | 8.47 | 12.9 |
| CatBoost(1000) | 1730 | 0.899 | 27.2 |


- The fastest (in training and prediction) model is linear regression. But the quality is low
- The model with the best quality is gradient boosting LightGBM with 1000 trees.
- The model with balanced prediction speed and quality is gradient boosting CatBoost with 1000 trees without category processing